In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('REPO_ROOT:', REPO_ROOT)

REPO_ROOT: C:\Users\ACER\OneDrive\Documents\final-project


## Section 1 — Caption central images with BLIP

In [2]:
import numpy as np
import pandas as pd
import json
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from utils.image_config import (
    IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES,
    IMAGE_NUM_CLASSES, BLIP_MODEL_NAME, RESULTS_DIR
)

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

print(f'Manifest shape: {manifest.shape}')
print(f'Embeddings shape: {embeddings.shape}')
print(f'BLIP model: {BLIP_MODEL_NAME}')

C:\python3.11.8\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Manifest shape: (5500, 4)
Embeddings shape: (5500, 512)
BLIP model: Salesforce/blip-image-captioning-base


In [3]:
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_NAME)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL_NAME)
blip_model.eval()

print('BLIP model loaded')

def caption_images(filepaths: list, max_new_tokens: int = 30) -> list:
    """Generate one caption per image path using BLIP."""
    captions = []
    for fp in filepaths:
        img = Image.open(fp).convert('RGB')
        inputs = blip_processor(images=img, return_tensors='pt')
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=max_new_tokens)
        captions.append(blip_processor.decode(out[0], skip_special_tokens=True))
    return captions

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 18053.89it/s]

BLIP model loaded


## Section 2 — For each method: find central images, caption them, build descriptions and label mappings

In [4]:
from utils.cluster_description import generate_cluster_descriptions
from utils.image_embeddings import get_clip_text_embeddings

CENTRAL_IMAGES_PER_CLUSTER = 8   # caption only 8 × 10 = 80 images total

def get_central_image_indices(embeddings, cluster_assignments, n_clusters, n_central=8):
    """Return dict {cluster_id: [row_indices of n_central images closest to centroid]}."""
    central = {}
    for c in range(n_clusters):
        mask = cluster_assignments == c
        if mask.sum() == 0:
            central[c] = []
            continue
        cluster_embs = embeddings[mask]
        centroid = cluster_embs.mean(axis=0)
        dists = np.linalg.norm(cluster_embs - centroid, axis=1)
        top_local = np.argsort(dists)[:n_central]
        global_indices = np.where(mask)[0][top_local]
        central[c] = global_indices.tolist()
    return central

In [5]:
for method in ['kmeans', 'agglomerative', 'dec']:
    print(f'\n=== Processing method: {method} ===')
    assignments = np.load(RESULTS_DIR / f'image_cluster_assignments_{method}.npy')
    print(f'  Assignments shape: {assignments.shape}, unique clusters: {len(np.unique(assignments))}')

    # Caption central images per cluster
    central_idx = get_central_image_indices(embeddings, assignments, IMAGE_NUM_CLASSES, CENTRAL_IMAGES_PER_CLUSTER)
    captions_by_cluster = {}
    for c, indices in central_idx.items():
        if not indices:
            captions_by_cluster[c] = []
            continue
        fps = manifest.iloc[indices]['filepath'].tolist()
        captions_by_cluster[c] = caption_images(fps)
        print(f'  {method} cluster {c}: {captions_by_cluster[c][:2]}')

    # Convert captions_by_cluster to flat lists for generate_cluster_descriptions
    summaries_flat = []
    labels_flat = []
    for c, caps in captions_by_cluster.items():
        summaries_flat.extend(caps)
        labels_flat.extend([c] * len(caps))

    # LexRank descriptions on captions (correct signature)
    descriptions = generate_cluster_descriptions(
        summaries_flat,
        np.array(labels_flat),
        n_sentences=2,
    )
    # descriptions keys are int cluster ids — convert to str for JSON
    descriptions_str = {str(c): desc for c, desc in descriptions.items()}
    with open(RESULTS_DIR / f'image_cluster_descriptions_{method}.json', 'w') as f:
        json.dump(descriptions_str, f, indent=2)
    print(f'  Saved descriptions for {method}')

    # CLIP zero-shot label matching
    class_prompts = [f'a photo of a {cls}' for cls in IMAGE_CLASS_NAMES]
    text_embs = get_clip_text_embeddings(class_prompts)  # (10, 512) L2-normalised

    mapping = {}
    for c, desc in descriptions.items():
        if not desc:
            # fallback: use cluster index to pick class
            mapping[str(c)] = IMAGE_CLASS_NAMES[c % IMAGE_NUM_CLASSES]
            continue
        desc_emb = get_clip_text_embeddings([desc])           # (1, 512)
        sims = (desc_emb @ text_embs.T).ravel()               # (10,)
        best_class = IMAGE_CLASS_NAMES[int(sims.argmax())]
        mapping[str(c)] = best_class

    with open(RESULTS_DIR / f'image_cluster_mapping_{method}.json', 'w') as f:
        json.dump(mapping, f, indent=2)
    print(f'{method} mapping: {mapping}')


=== Processing method: kmeans ===
  Assignments shape: (5500,), unique clusters: 10


  kmeans cluster 0: ['a cow grazing in a field', 'a group of cows grazing in a field']


  kmeans cluster 1: ['a spider is hanging on the side of a window', 'a spider with a long tail and a long tail']


  kmeans cluster 2: ['a cat laying on a carpet', 'a fluffy white cat']


  kmeans cluster 3: ['a squirrel is sitting on the ground', 'a squirrel is sitting on the ground']


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 17126.81it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 18789.48it/s]

dec mapping: {'0': 'butterfly', '1': 'dog', '2': 'horse', '3': 'chicken', '4': 'squirrel', '5': 'elephant', '6': 'cow', '7': 'spider', '8': 'cat', '9': 'sheep'}


## Verification — Confirm all 6 JSON files exist

In [6]:
expected_files = [
    'image_cluster_descriptions_kmeans.json',
    'image_cluster_descriptions_agglomerative.json',
    'image_cluster_descriptions_dec.json',
    'image_cluster_mapping_kmeans.json',
    'image_cluster_mapping_agglomerative.json',
    'image_cluster_mapping_dec.json',
]

all_ok = True
for fname in expected_files:
    fpath = RESULTS_DIR / fname
    exists = fpath.exists()
    size = fpath.stat().st_size if exists else 0
    status = 'OK' if exists and size > 0 else 'MISSING or EMPTY'
    print(f'  {fname}: {status} ({size} bytes)')
    if not (exists and size > 0):
        all_ok = False

print('\nAll 6 JSON files created:', all_ok)

  image_cluster_descriptions_kmeans.json: OK (3252 bytes)
  image_cluster_descriptions_agglomerative.json: OK (3252 bytes)
  image_cluster_descriptions_dec.json: OK (3252 bytes)
  image_cluster_mapping_kmeans.json: OK (180 bytes)
  image_cluster_mapping_agglomerative.json: OK (180 bytes)
  image_cluster_mapping_dec.json: OK (180 bytes)

All 6 JSON files created: True
